### Project Solution

This notebook solves the NYC parking tickets extract project using lazy iteration, typed parsing, named tuples, and an efficient aggregation step.

### Goals

1. Create a lazy iterator that yields a named tuple for each row.
2. Convert values to appropriate Python types.
3. Calculate the number of violations by vehicle make.

The iterator is lazy: it reads and parses one CSV row at a time instead of loading the whole file into memory. The final counting step must keep counts in memory, but it still consumes the source rows lazily.

In [1]:
from __future__ import annotations

import csv
from collections import Counter, namedtuple
from datetime import datetime, date
from itertools import islice
from pathlib import Path
from typing import Iterable, Iterator, Mapping

### Configuration

Place `nyc_parking_tickets_extract.csv` in the same folder as this notebook, or update `CSV_FILE` to the correct path.

In [2]:
CSV_FILE = Path("nyc_parking_tickets_extract.csv")

# Expected columns in the sample file. These are kept as the original CSV names
# so that validation remains clear and error messages are easy to understand.
EXPECTED_COLUMNS = (
    "Summons Number",
    "Plate ID",
    "Registration State",
    "Plate Type",
    "Issue Date",
    "Violation Code",
    "Vehicle Body Type",
    "Vehicle Make",
    "Violation Description",
)

### Named tuple model

The original CSV headers contain spaces and title case. For Python code, we use clean snake-case field names in the named tuple.

In [3]:
ParkingTicket = namedtuple(
    "ParkingTicket",
    [
        "summons_number",
        "plate_id",
        "registration_state",
        "plate_type",
        "issue_date",
        "violation_code",
        "vehicle_body_type",
        "vehicle_make",
        "violation_description",
    ],
)

ParkingTicket.__doc__ = "Typed NYC parking ticket violation record."

### Parsing helpers

These helper functions keep parsing explicit and testable. Empty strings are normalized to `None` for optional text-like fields. Required integer and date fields raise clear errors if parsing fails.

In [4]:
def clean_text(value: str | None) -> str | None:
    """Return stripped text, or None when the CSV cell is empty."""
    if value is None:
        return None
    value = value.strip()
    return value or None


def parse_int(value: str | None, *, field_name: str) -> int:
    """Parse a required integer field with a helpful error message."""
    value = clean_text(value)
    if value is None:
        raise ValueError(f"Missing required integer value for {field_name!r}")
    try:
        return int(value)
    except ValueError as exc:
        raise ValueError(f"Invalid integer for {field_name!r}: {value!r}") from exc


def parse_date(value: str | None, *, field_name: str, date_format: str = "%m/%d/%Y") -> date:
    """Parse a required date field from the NYC sample format: MM/DD/YYYY."""
    value = clean_text(value)
    if value is None:
        raise ValueError(f"Missing required date value for {field_name!r}")
    try:
        return datetime.strptime(value, date_format).date()
    except ValueError as exc:
        raise ValueError(
            f"Invalid date for {field_name!r}: {value!r}; expected format {date_format!r}"
        ) from exc


def normalize_make(value: str | None, *, unknown_label: str = "UNKNOWN") -> str:
    """Normalize vehicle make for counting."""
    value = clean_text(value)
    return value.upper() if value is not None else unknown_label

### Row parser

`parse_ticket_row` converts one raw CSV row into one typed `ParkingTicket`. This function is intentionally separate from the file iterator so it can be tested independently.

In [5]:
def parse_ticket_row(row: Mapping[str, str | None]) -> ParkingTicket:
    """Convert a raw CSV row mapping into a typed ParkingTicket named tuple."""
    return ParkingTicket(
        summons_number=parse_int(row.get("Summons Number"), field_name="Summons Number"),
        plate_id=clean_text(row.get("Plate ID")),
        registration_state=clean_text(row.get("Registration State")),
        plate_type=clean_text(row.get("Plate Type")),
        issue_date=parse_date(row.get("Issue Date"), field_name="Issue Date"),
        violation_code=parse_int(row.get("Violation Code"), field_name="Violation Code"),
        vehicle_body_type=clean_text(row.get("Vehicle Body Type")),
        vehicle_make=normalize_make(row.get("Vehicle Make")),
        violation_description=clean_text(row.get("Violation Description")),
    )

### CSV validation

A small validation step catches common mistakes early, such as pointing to the wrong file or using a CSV with different headers.

In [6]:
def validate_columns(actual_columns: Iterable[str] | None, expected_columns: Iterable[str]) -> None:
    """Validate that the CSV has all expected columns."""
    if actual_columns is None:
        raise ValueError("CSV file has no header row.")

    actual = tuple(actual_columns)
    expected = tuple(expected_columns)

    missing = [column for column in expected if column not in actual]
    if missing:
        raise ValueError(
            "CSV file is missing expected columns: "
            + ", ".join(repr(column) for column in missing)
        )

### Goal 1: lazy iterator

This generator opens the file, validates the header, and yields one typed named tuple per row. Because it is a generator, rows are parsed only when requested.

In [7]:
def iter_parking_tickets(
    file_path: str | Path = CSV_FILE,
    *,
    encoding: str = "utf-8",
    skip_bad_rows: bool = False,
) -> Iterator[ParkingTicket]:
    """
    Lazily yield typed ParkingTicket records from the NYC parking ticket CSV.

    Parameters
    ----------
    file_path:
        Path to the CSV file.
    encoding:
        File encoding. Use "utf-8-sig" if your CSV includes a BOM.
    skip_bad_rows:
        If False, fail fast on malformed rows. If True, malformed rows are skipped.

    Yields
    ------
    ParkingTicket
        One typed named tuple per CSV row.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path!s}. Put the CSV next to this notebook or update CSV_FILE."
        )

    with path.open(mode="r", newline="", encoding=encoding) as file:
        reader = csv.DictReader(file)
        validate_columns(reader.fieldnames, EXPECTED_COLUMNS)

        for row_number, row in enumerate(reader, start=2):
            try:
                yield parse_ticket_row(row)
            except Exception as exc:
                if skip_bad_rows:
                    continue
                raise ValueError(f"Failed to parse CSV row {row_number}") from exc

### Sanity check: preview a few records

`islice` lets us inspect a few rows without materializing the full file.

In [8]:
sample_records = list(islice(iter_parking_tickets(), 5))
sample_records

[ParkingTicket(summons_number=4006478550, plate_id='VAD7274', registration_state='VA', plate_type='PAS', issue_date=datetime.date(2016, 10, 5), violation_code=5, vehicle_body_type='4D', vehicle_make='BMW', violation_description='BUS LANE VIOLATION'),
 ParkingTicket(summons_number=4006462396, plate_id='22834JK', registration_state='NY', plate_type='COM', issue_date=datetime.date(2016, 9, 30), violation_code=5, vehicle_body_type='VAN', vehicle_make='CHEVR', violation_description='BUS LANE VIOLATION'),
 ParkingTicket(summons_number=4007117810, plate_id='21791MG', registration_state='NY', plate_type='COM', issue_date=datetime.date(2017, 4, 10), violation_code=5, vehicle_body_type='VAN', vehicle_make='DODGE', violation_description='BUS LANE VIOLATION'),
 ParkingTicket(summons_number=4006265037, plate_id='FZX9232', registration_state='NY', plate_type='PAS', issue_date=datetime.date(2016, 8, 23), violation_code=5, vehicle_body_type='SUBN', vehicle_make='FORD', violation_description='BUS LANE 

Check the parsed Python types.

In [9]:
if sample_records:
    first = sample_records[0]
    type_report = {field: type(getattr(first, field)).__name__ for field in first._fields}
    type_report

### Goal 2: violations by car make

Counting requires an accumulator, so this part cannot be completely lazy. However, the input stream is still consumed lazily: only the `Counter` is kept in memory.

In [10]:
def violations_by_make(records: Iterable[ParkingTicket]) -> Counter[str]:
    """Return a Counter mapping vehicle make to number of violations."""
    return Counter(record.vehicle_make for record in records)


make_counts = violations_by_make(iter_parking_tickets())
make_counts

Counter({'TOYOT': 112,
         'HONDA': 106,
         'FORD': 104,
         'CHEVR': 76,
         'NISSA': 70,
         'DODGE': 45,
         'FRUEH': 44,
         'ME/BE': 38,
         'GMC': 35,
         'HYUND': 35,
         'BMW': 34,
         'LEXUS': 26,
         'INTER': 25,
         'JEEP': 22,
         'NS/OT': 18,
         'SUBAR': 18,
         'INFIN': 13,
         'LINCO': 12,
         'CHRYS': 12,
         'ACURA': 12,
         'AUDI': 12,
         'VOLVO': 12,
         'MITSU': 11,
         'ISUZU': 10,
         'CADIL': 9,
         'KIA': 8,
         'VOLKS': 8,
         'HIN': 6,
         'KENWO': 5,
         'UNKNOWN': 5,
         'ROVER': 5,
         'BUICK': 5,
         'MAZDA': 5,
         'MERCU': 4,
         'JAGUA': 3,
         'SMART': 3,
         'PORSC': 3,
         'WORKH': 2,
         'SATUR': 2,
         'SCION': 2,
         'SAAB': 2,
         'HINO': 2,
         'FIR': 1,
         'OLDSM': 1,
         'PETER': 1,
         'CITRO': 1,
         'GEO': 1,
 

### Display results sorted by frequency

`Counter.most_common()` gives a clean descending ranking.

In [11]:
make_counts.most_common()

[('TOYOT', 112),
 ('HONDA', 106),
 ('FORD', 104),
 ('CHEVR', 76),
 ('NISSA', 70),
 ('DODGE', 45),
 ('FRUEH', 44),
 ('ME/BE', 38),
 ('GMC', 35),
 ('HYUND', 35),
 ('BMW', 34),
 ('LEXUS', 26),
 ('INTER', 25),
 ('JEEP', 22),
 ('NS/OT', 18),
 ('SUBAR', 18),
 ('INFIN', 13),
 ('LINCO', 12),
 ('CHRYS', 12),
 ('ACURA', 12),
 ('AUDI', 12),
 ('VOLVO', 12),
 ('MITSU', 11),
 ('ISUZU', 10),
 ('CADIL', 9),
 ('KIA', 8),
 ('VOLKS', 8),
 ('HIN', 6),
 ('KENWO', 5),
 ('UNKNOWN', 5),
 ('ROVER', 5),
 ('BUICK', 5),
 ('MAZDA', 5),
 ('MERCU', 4),
 ('JAGUA', 3),
 ('SMART', 3),
 ('PORSC', 3),
 ('WORKH', 2),
 ('SATUR', 2),
 ('SCION', 2),
 ('SAAB', 2),
 ('HINO', 2),
 ('FIR', 1),
 ('OLDSM', 1),
 ('PETER', 1),
 ('CITRO', 1),
 ('GEO', 1),
 ('YAMAH', 1),
 ('BSA', 1),
 ('MINI', 1),
 ('PONTI', 1),
 ('SPRI', 1),
 ('PLYMO', 1),
 ('UPS', 1),
 ('FIAT', 1),
 ('UD', 1),
 ('UTILI', 1),
 ('GMCQ', 1),
 ('STAR', 1),
 ('AM/T', 1),
 ('MI/F', 1)]

### Optional: tabular display with pandas

This is optional and only used for presentation. The core solution above does not require pandas.

In [12]:
try:
    import pandas as pd

    make_counts_df = pd.DataFrame(
        make_counts.most_common(),
        columns=["vehicle_make", "violation_count"],
    )
    display(make_counts_df)
except ImportError:
    print("pandas is not installed; showing plain Python output instead.")
    print(make_counts.most_common())

pandas is not installed; showing plain Python output instead.
[('TOYOT', 112), ('HONDA', 106), ('FORD', 104), ('CHEVR', 76), ('NISSA', 70), ('DODGE', 45), ('FRUEH', 44), ('ME/BE', 38), ('GMC', 35), ('HYUND', 35), ('BMW', 34), ('LEXUS', 26), ('INTER', 25), ('JEEP', 22), ('NS/OT', 18), ('SUBAR', 18), ('INFIN', 13), ('LINCO', 12), ('CHRYS', 12), ('ACURA', 12), ('AUDI', 12), ('VOLVO', 12), ('MITSU', 11), ('ISUZU', 10), ('CADIL', 9), ('KIA', 8), ('VOLKS', 8), ('HIN', 6), ('KENWO', 5), ('UNKNOWN', 5), ('ROVER', 5), ('BUICK', 5), ('MAZDA', 5), ('MERCU', 4), ('JAGUA', 3), ('SMART', 3), ('PORSC', 3), ('WORKH', 2), ('SATUR', 2), ('SCION', 2), ('SAAB', 2), ('HINO', 2), ('FIR', 1), ('OLDSM', 1), ('PETER', 1), ('CITRO', 1), ('GEO', 1), ('YAMAH', 1), ('BSA', 1), ('MINI', 1), ('PONTI', 1), ('SPRI', 1), ('PLYMO', 1), ('UPS', 1), ('FIAT', 1), ('UD', 1), ('UTILI', 1), ('GMCQ', 1), ('STAR', 1), ('AM/T', 1), ('MI/F', 1)]


### Optional: fully reusable pipeline

This wrapper makes the solution easy to reuse from another notebook cell or script.

In [13]:
def solve(file_path: str | Path = CSV_FILE) -> Counter[str]:
    """Run the complete solution and return violations counted by vehicle make."""
    return violations_by_make(iter_parking_tickets(file_path))


solution_counts = solve()
solution_counts.most_common(10)

[('TOYOT', 112),
 ('HONDA', 106),
 ('FORD', 104),
 ('CHEVR', 76),
 ('NISSA', 70),
 ('DODGE', 45),
 ('FRUEH', 44),
 ('ME/BE', 38),
 ('GMC', 35),
 ('HYUND', 35)]

### Notes on laziness and memory use

- `iter_parking_tickets()` is lazy and reads one row at a time.
- `parse_ticket_row()` converts only the current row.
- `Counter` is the only necessary eager structure because counting frequencies requires remembering the counts seen so far.
- The solution avoids `list(...)` except for the tiny preview sample.